<a href="https://colab.research.google.com/github/Hafsahshaikh/Multilingual_Text_to_Speech/blob/main/Multilingual_Text_to_Speech_HuggingFace.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multilingual Text-to-Speech with Hugging Face

Generate speech in **English, Hindi, Telugu, and Turkish** in Google Colab using pretrained Hugging Face MMS-TTS models.

**Note:** This is text-to-speech, not translation. The linked model checkpoints are marked CC-BY-NC 4.0 on their model cards. Check each model card for current terms and attribution requirements; do not use commercially without appropriate permission.

## 1. Install dependencies

In [ ]:
pip install transformers torch soundfile

In [2]:
%pip install --force-reinstall torch==2.11.0 torchvision==0.26.0

  Using cached cuda_bindings-13.4.4-cp313-cp313-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (2.4 kB)
  Using cached cuda_pathfinder-1.8.4-py3-none-any.whl.metadata (1.9 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 14.2 MB/s  0:00:24
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 58.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 12.3 MB/s  0:00:22
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 12.2 MB/s  0:00:13
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 11.7 MB/s  0:00:16
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 13.6 MB/s  0:00:04
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 10.7 MB/s  0:00:17
Using cached cuda_bindings-13.4.4-cp313-cp313-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (6.9 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 13.2 MB/s  0:00:26
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 25.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━

## 2. Imports and runtime check

In [11]:
import torch
import numpy as np
import soundfile as sf
from IPython.display import Audio, display
from transformers import AutoTokenizer, VitsModel

print("PyTorch:", torch.__version__)
print("TorchVision:", torchvision.__version__)
print("Transformers:", transformers.__version__)
print("VITS imports successful!")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)
print("NumPy version:", np.__version__)
print("All imports successful!")

PyTorch: 2.11.0+cu130
TorchVision: 0.26.0+cu130
Transformers: 5.21.1
VITS imports successful!
Device: cpu
NumPy version: 2.5.4
All imports successful!


## 3. Language models and sample sentences

In [2]:
MODEL_IDS = {
    "English": "facebook/mms-tts-eng",
    "Hindi": "facebook/mms-tts-hin",
    "Telugu": "facebook/mms-tts-tel",
    "Turkish": "facebook/mms-tts-tur",
}

EXAMPLES = {
    "English": "Hello! Welcome to my multilingual text-to-speech project.",
    "Hindi": "नमस्ते! मेरे बहुभाषी टेक्स्ट-टू-स्पीच प्रोजेक्ट में आपका स्वागत है।",
    "Telugu": "నమస్కారం! నా బహుభాషా టెక్స్ట్ టు స్పీచ్ ప్రాజెక్ట్‌కు స్వాగతం.",
    "Turkish": "Merhaba! Çok dilli metinden konuşmaya projeme hoş geldiniz.",
}

for language, model_id in MODEL_IDS.items():
    print(f"{language}: https://huggingface.co/{model_id}")


English: https://huggingface.co/facebook/mms-tts-eng
Hindi: https://huggingface.co/facebook/mms-tts-hin
Telugu: https://huggingface.co/facebook/mms-tts-tel
Turkish: https://huggingface.co/facebook/mms-tts-tur


## 4. Load and cache the selected model

In [3]:
_model_cache = {}

def load_tts_model(language):
    if language not in MODEL_IDS:
        raise ValueError(f"Choose one of: {list(MODEL_IDS)}")
    if language not in _model_cache:
        model_id = MODEL_IDS[language]
        print(f"Loading {language}: {model_id}")
        tokenizer = AutoTokenizer.from_pretrained(model_id)
        model = VitsModel.from_pretrained(model_id).to(DEVICE).eval()
        _model_cache[language] = (tokenizer, model)
    return _model_cache[language]


## 5. Generate and save speech

In [9]:
def generate_speech(text, language, output_path=None):
    """Generate speech and save it as a WAV file.
    Returns (output_path, waveform, sample_rate).
    """
    if not isinstance(text, str) or not text.strip():
        raise ValueError("Enter some text before generating speech.")
    tokenizer, model = load_tts_model(language)
    inputs = tokenizer(text.strip(), return_tensors="pt")
    inputs = {key: value.to(DEVICE) for key, value in inputs.items()}
    with torch.inference_mode():
        waveform_tensor = model(**inputs).waveform
    waveform = waveform_tensor.squeeze().detach().cpu().numpy().astype(np.float32)
    sample_rate = int(model.config.sampling_rate)
    if output_path is None:
        output_path = f"speech_{language.lower()}.wav"
    sf.write(output_path, waveform, sample_rate)
    return output_path, waveform, sample_rate


## 6. Run a sample

Choose a language and enter text in that language. This notebook does not translate text.

In [12]:
language = "English"  # English, Hindi, Telugu, or Turkish
text = EXAMPLES[language]  # Replace with your own sentence

output_path, waveform, sample_rate = generate_speech(text, language)
print("Saved:", output_path)
print("Sample rate:", sample_rate, "Hz")
display(Audio(waveform, rate=sample_rate))


Saved: speech_english.wav
Sample rate: 16000 Hz


## 7. Speech-quality evaluation

This section adds an **experimental overall MOS prediction** plus simple waveform checks. It is inspired by the idea of evaluating generated speech.

Run these cells after generating `output_path` above. CAL-MOS was trained on the BVCC dataset; predictions for Hindi, Telugu, and Turkish may be out of distribution and must not be treated as validated language-specific scores. Compare with human listening ratings.


### 7A. Basic acoustic checks

These measurements can flag possible issues, but they do not directly measure naturalness or intelligibility.


In [13]:
import numpy as np
import soundfile as sf

def inspect_audio(path, near_zero_threshold=1e-4, clipping_threshold=0.999):
    audio, sr = sf.read(path)
    if audio.ndim == 2:
        audio = audio.mean(axis=1)
    duration = len(audio) / sr if sr else 0.0
    peak = float(np.max(np.abs(audio))) if len(audio) else 0.0
    rms = float(np.sqrt(np.mean(np.square(audio)))) if len(audio) else 0.0
    near_zero_pct = float(np.mean(np.abs(audio) < near_zero_threshold) * 100) if len(audio) else 0.0
    clipped_pct = float(np.mean(np.abs(audio) >= clipping_threshold) * 100) if len(audio) else 0.0
    report = {
        "file": path,
        "sample_rate_hz": int(sr),
        "duration_seconds": round(duration, 2),
        "peak_amplitude": round(peak, 4),
        "rms_amplitude": round(rms, 4),
        "near_silence_samples_percent": round(near_zero_pct, 2),
        "near_full_scale_samples_percent": round(clipped_pct, 4),
    }
    for key, value in report.items():
        print(f"{key}: {value}")
    print("Note: thresholds are rough diagnostics, not perceptual quality scores.")
    return report

audio_report = inspect_audio(output_path)


file: speech_english.wav
sample_rate_hz: 16000
duration_seconds: 4.02
peak_amplitude: 0.7711
rms_amplitude: 0.1049
near_silence_samples_percent: 14.98
near_full_scale_samples_percent: 0.0
Note: thresholds are rough diagnostics, not perceptual quality scores.


### 7B. Predict an overall MOS score with CAL-MOS (optional)

This downloads an additional model and may take time or memory. The checkpoint uses WavLM-Large and is documented for 16 kHz input; its `predict()` method handles audio conversion/resampling.


In [14]:
# Optional: run this cell if you want a learned overall MOS estimate.
# The model repository uses custom model code, so trust_remote_code=True is required.
from transformers import AutoModel

MOS_MODEL_ID = "alefiury/CALMOS-WavLM-large-BVCC"
mos_model = AutoModel.from_pretrained(
    MOS_MODEL_ID,
    trust_remote_code=True
).to(DEVICE)
mos_model.eval()

mos_prediction = mos_model.predict(output_path)
print("CAL-MOS predicted overall MOS (nominal range 1–5):", mos_prediction)
print("Interpret cautiously, especially for Hindi, Telugu, and Turkish; validate against human ratings.")


config.json:   0%|          | 0.00/589 [00:00<?, ?B/s]

configuration_calmos.py:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/alefiury/CALMOS-WavLM-large-BVCC:
- configuration_calmos.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_calmos.py:   0%|          | 0.00/20.4k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/alefiury/CALMOS-WavLM-large-BVCC:
- modeling_calmos.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


...

model.safetensors: reconstructing file:   0%|          |  0.00B /  214MB            

model.safetensors: downloading bytes:           |  0.00B            

preprocessor_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


CAL-MOS predicted overall MOS (nominal range 1–5): 3.8662109375
Interpret cautiously, especially for Hindi, Telugu, and Turkish; validate against human ratings.


### 7C. Human ratings for the four dimensions

Listen to the generated clip and give each dimension a rating from 1 (poor) to 5 (excellent). For stronger evaluation, ask multiple listeners to rate the same clips and average their scores.


In [15]:
from IPython.display import Audio, display
from IPython.display import Markdown

display(Audio(filename=output_path))
print("Rate each dimension from 1 (poor) to 5 (excellent).")
acoustic_rating = int(input("Acoustic clarity/fidelity (1-5): "))
prosody_rating = int(input("Naturalness of rhythm and intonation (1-5): "))
intelligibility_rating = int(input("Pronunciation / intelligibility (1-5): "))
semantic_rating = int(input("Does the speech communicate the intended text clearly? (1-5): "))

human_ratings = {
    "language": language,
    "input_text": text,
    "acoustic_fidelity_1_to_5": acoustic_rating,
    "prosodic_naturalness_1_to_5": prosody_rating,
    "phonetic_intelligibility_1_to_5": intelligibility_rating,
    "semantic_clarity_1_to_5": semantic_rating,
}
print("Human listening ratings:", human_ratings)


Rate each dimension from 1 (poor) to 5 (excellent).
Acoustic clarity/fidelity (1-5): 3
Naturalness of rhythm and intonation (1-5): 1
Pronunciation / intelligibility (1-5): 3
Does the speech communicate the intended text clearly? (1-5): 3
Human listening ratings: {'language': 'English', 'input_text': 'Hello! Welcome to my multilingual text-to-speech project.', 'acoustic_fidelity_1_to_5': 3, 'prosodic_naturalness_1_to_5': 1, 'phonetic_intelligibility_1_to_5': 3, 'semantic_clarity_1_to_5': 3}


## 8. Download the WAV file


In [17]:
from google.colab import files
files.download(output_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## How it works

1. Language selection chooses a matching Hugging Face model.
2. The tokenizer converts text into model inputs.
3. The VITS model generates a numerical speech waveform.
4. SoundFile saves the waveform as WAV and Colab previews it.
5. Optional evaluation checks waveform statistics, predicts an experimental overall MOS score, and collects human ratings for four quality dimensions.

## Troubleshooting

- **Slow first run:** model weights are downloading.
- **Memory error:** restart the runtime and test one language at a time.
- **Pronunciation issue:** check that text and selected language match.
- **MOS model loading error:** skip the optional CAL-MOS cell and continue with the waveform checks and human ratings.
- **Model/API error:** check the model card and rerun cells after restarting the runtime.
